# Internet use and global D1

This notebook compares the share of each country's population in the global bottom decile with the percentage of individuals using the internet. The deprivation proxy is `100 - internet use percentage`.

## Source choice

We use **`ITU_DH / IT_NET_USER`**, the ITU series, because it is the primary source and has the widest coverage: 183 countries in 2024, accounting for 90.6% of the global D1.

Data360 carries three other internet series, all of which trace back to ITU, so none of them provides an independent cross-check:

| Series | Countries (latest) | Share of global D1 covered | Note |
| --- | --- | --- | --- |
| `ITU_DH / IT_NET_USER` | 183 | 90.6% | Primary source, used here |
| `WB_WDI / WB_WDI_IT_NET_USER_ZS` | 182 | 90.6% | Identical to ITU in all 4,898 shared observations |
| `WB_CSC / WB_CSC_IT_NET_USER_ZS` | 127 | 73.0% | Matches ITU in 90% of observations; omits Ethiopia and India |
| `WEF_TTDI / WEF_TTDI_NETUSERPCT` | 119 | 52.5% | Excludes DR Congo, Ethiopia and Mozambique; only 3 years |

Coverage of the D1 drives the result: the same analysis yields a 2024 gap ratio of 4.93x with ITU, 3.17x with `WB_CSC` and 1.76x with `WEF_TTDI`, because the thinner samples omit the countries where the poorest decile actually lives.

In [17]:
from pathlib import Path
import sys
import importlib
import altair as alt
import pandas as pd

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
import indicator_analysis_helpers
importlib.reload(indicator_analysis_helpers)
from indicator_analysis_helpers import REGION_COLORS, run_indicator_analysis

result = run_indicator_analysis('ITU_DH', 'IT_NET_USER', 'coverage_pct')
coverage_by_year = result['joined'].groupby('year')['code'].nunique()
reliable_years = coverage_by_year[coverage_by_year >= 100].index
last_reliable_year = int(reliable_years.max())
plot_df = result['joined'][result['joined']['year'].isin([result['first_year'], last_reliable_year])].copy()
plot_df = plot_df.dropna(subset=['region'])
plot_df['year'] = plot_df['year'].astype(str)
plot_years = [str(result['first_year']), str(last_reliable_year)]

In [ ]:
first_year = result['first_year']
latest_year = result['last_year']
latest_country_count = result['joined'].loc[result['joined']['year'].eq(latest_year), 'code'].nunique()
comparison_years = [first_year, last_reliable_year]
print(f"Comparison years: {first_year} and {last_reliable_year}")
print(result['deprivation_label'])
print(f"Warning: raw latest year {latest_year} is sparse ({latest_country_count} countries); using {last_reliable_year} as the comparison year.")
display(result['correlations'].loc[result['correlations']['year'].isin(comparison_years)].round(3))
comparison_rows = []
for year, group in result['joined'][result['joined']['year'].isin(comparison_years)].groupby('year'):
    comparison_rows.append({
        'year': year,
        'countries': group['code'].nunique(),
        'population_weighted_indicator': (group['indicator_value'] * group['country_population']).sum() / group['country_population'].sum(),
        'population_weighted_deprivation_proxy': (group['deprivation_proxy'] * group['country_population']).sum() / group['country_population'].sum(),
    })
comparison_summary = pd.DataFrame(comparison_rows).sort_values('year')
display(comparison_summary.round(3))

Comparison years: 2000 and 2024
Estimated population without access (%) = 100 - coverage


,year,pearson_r,spearman_r,observations
0,2000,0.355,0.742,197
24,2024,0.861,0.846,183


,year,countries,population_weighted_indicator,population_weighted_deprivation_proxy,d1_weighted_deprivation_proxy
0,2000,197,6.736,93.264,98.638
1,2024,183,72.734,27.266,62.657


## Scatter plot: internet use and global D1

Each bubble is a country. Color is the World Bank region, the horizontal axis is the percentage of the country's population in the global D1, the vertical axis is internet use, and bubble size reflects population.

In [9]:
point_chart = (
    alt.Chart(plot_df)
    .mark_circle(opacity=0.7, stroke='white', strokeWidth=0.5)
    .encode(
        x=alt.X('d1_country_population_pct:Q', title='Population in global D1 (%)'),
        y=alt.Y('indicator_value:Q', title='Individuals using the internet (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('region:N', title='World Bank region', scale=alt.Scale(domain=list(REGION_COLORS), range=list(REGION_COLORS.values()))),
        size=alt.Size('country_population:Q', legend=None, scale=alt.Scale(range=[40, 900])),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('d1_country_population_pct:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('indicator_value:Q', title='Internet use (%)', format='.1f'),
        ],
    )
    .properties(width=420, height=420)
)
internet_scatter = point_chart.facet(column=alt.Column('year:O', title=None, sort=plot_years)).properties(title='Internet use versus population in the global D1')
internet_scatter

alt.FacetChart(...)

## Global population-weighted trend

This line uses country population weights, so larger countries contribute more to the global estimate.

In [12]:
trend_rows = []
for year, group in result['joined'].groupby('year'):
    trend_rows.append({
        'year': year,
        'global_access_pct': (group['indicator_value'] * group['country_population']).sum() / group['country_population'].sum(),
        'global_without_access_pct': (group['deprivation_proxy'] * group['country_population']).sum() / group['country_population'].sum(),
        'countries_with_data': group['code'].nunique(),
    })
global_trend = pd.DataFrame(trend_rows).sort_values('year')
trend_line = (
    alt.Chart(global_trend)
    .mark_line(color='#0072B2', strokeWidth=3)
    .encode(
        x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d')),
        y=alt.Y('global_without_access_pct:Q', title='Population without internet access (%)'),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('global_without_access_pct:Q', title='Without access (%)', format='.2f'),
            alt.Tooltip('global_access_pct:Q', title='Internet use (%)', format='.2f'),
            alt.Tooltip('countries_with_data:Q', title='Countries with data'),
        ],
    )
    .properties(width=820, height=380, title='Global population without internet access over time')
)
trend_line

alt.Chart(...)

## Where the poorest decile lives

This measure only counts **where people live**, so it makes no assumption about who within a country has access.

For each year it reports the share of the global poorest decile living in countries where fewer than half the population uses the internet, next to the same share for the world population. The ratio between the two lines is the gap.

In [14]:
from indicator_analysis_helpers import low_coverage_chart

low_coverage = result['low_coverage']
display(low_coverage[low_coverage['year'].isin(comparison_years)].round(2))
print(f"Gap ratio: {low_coverage.iloc[0]['gap_ratio']:.2f}x in {low_coverage.iloc[0]['year']}"
      f" -> {low_coverage.iloc[-1]['gap_ratio']:.2f}x in {low_coverage.iloc[-1]['year']}")
low_coverage_chart(low_coverage, 'Internet use')

,year,d1_share_below,world_share_below,gap_ratio,countries_below,countries_with_data
0,2000,99.99,99.40,1.01,195,197
24,2024,74.00,15.02,4.93,38,183


Gap ratio: 1.01x in 2000.0 -> 4.93x in 2024.0


alt.Chart(...)

## Interpretation

All three measures here are statements about countries, not about households.

- The **scatter plot** and the **correlations** compare countries: internet use is systematically lower where a larger share of the population falls in the global D1.
- The **global trend** is the population-weighted world average.
- The **low-coverage concentration** counts where the poorest decile lives, which is directly observable.

What these data cannot show is the internet use of the poorest decile itself. That would require household microdata linking income to access. Any country-level figure would have to assume that the poor within a country have the same access as the national average, which is false and would understate deprivation.

The correlations describe association, not causation.